
# Unit 1 — Mendelian Genetics Simulator (v3)

This Colab notebook lets you:
- Simulate **monohybrid** and **dihybrid** crosses
- Generate Punnett squares (1-locus and 2-locus)
- Run chi-square goodness-of-fit for observed counts vs expected

**How to use:** Run each code cell in order. Cells with a form header (they contain `#@title`) expose interactive controls — editable parameters use `#@param` annotations (e.g., `n_offspring = 1000 #@param {type:"integer"}`). Use those controls and re-run the cell to update results.


In [ ]:
#@title Imports and helper functions { display-mode: "form" }
import random
import pandas as pd
import numpy as np
from collections import Counter
import math
import itertools

def phenotype_from_genotype(genotype, trait_rules=None):
    if trait_rules is None:
        trait_rules = {'dominant_label':'Dominant', 'recessive_label':'Recessive'}
    if any(allele.isupper() for allele in genotype):
        return trait_rules.get('dominant_label','Dominant')
    else:
        return trait_rules.get('recessive_label','Recessive')

def chi_square(observed, expected):
    obs = np.array(observed, dtype=float)
    exp = np.array(expected, dtype=float)
    chisq = np.sum((obs - exp)**2 / exp)
    dof = len(observed) - 1
    return chisq, dof

def split_loci(genotype):
    if len(genotype) % 2 != 0:
        raise ValueError("Genotype length should be even (pairs of alleles per locus).")
    loci = []
    for i in range(0, len(genotype), 2):
        loci.append((genotype[i], genotype[i+1]))
    return loci

def gametes_from_parent(genotype):
    loci = split_loci(genotype)
    gametes = list(itertools.product(*loci))
    return gametes

def simulate_monohybrid(parent1_genotype, parent2_genotype, n_offspring=1000):
    gametes_p1 = [parent1_genotype[0], parent1_genotype[1]]
    gametes_p2 = [parent2_genotype[0], parent2_genotype[1]]
    offspring_genotypes = []
    for _ in range(n_offspring):
        g1 = random.choice(gametes_p1)
        g2 = random.choice(gametes_p2)
        offspring = ''.join(sorted([g1, g2], key=lambda x: x.islower()))
        offspring_genotypes.append(offspring)
    counts = Counter(offspring_genotypes)
    return counts

def punnett_square(parent1, parent2):
    g1 = [parent1[0], parent1[1]]
    g2 = [parent2[0], parent2[1]]
    table = pd.DataFrame(index=g1, columns=g2)
    for a in g1:
        for b in g2:
            genotype = ''.join(sorted([a,b], key=lambda x: x.islower()))
            table.loc[a,b] = genotype
    return table

def simulate_dihybrid(parent1_genotype, parent2_genotype, n_offspring=10000):
    gametes_p1 = gametes_from_parent(parent1_genotype)
    gametes_p2 = gametes_from_parent(parent2_genotype)
    offspring = []
    for _ in range(n_offspring):
        g1 = random.choice(gametes_p1)
        g2 = random.choice(gametes_p2)
        loci_pairs = []
        for a,b in zip(g1,g2):
            loci_pairs.append(''.join(sorted([a,b], key=lambda x: x.islower())))
        genotype = ''.join(loci_pairs)
        offspring.append(genotype)
    return Counter(offspring)

def punnett_square_dihybrid(parent1, parent2):
    gam1 = [''.join(g) for g in gametes_from_parent(parent1)]
    gam2 = [''.join(g) for g in gametes_from_parent(parent2)]
    table = pd.DataFrame(index=gam1, columns=gam2)
    for a in gam1:
        for b in gam2:
            loci_pairs = []
            for x,y in zip(a,b):
                loci_pairs.append(''.join(sorted([x,y], key=lambda z: z.islower())))
            table.loc[a,b] = ''.join(loci_pairs)
    return table

def phenotype_dihybrid(genotype, dominance='classical'):
    loci = split_loci(genotype)
    labels = []
    for pair in loci:
        if dominance == 'classical':
            if any(allele.isupper() for allele in pair):
                labels.append('Dominant')
            else:
                labels.append('Recessive')
        else:
            s = ''.join(pair)
            if s.isupper(): labels.append('HomoDominant')
            elif s.islower(): labels.append('HomoRecessive')
            else: labels.append('Intermediate')
    return '_'.join(labels)

def phenotype_counts_from_genotype_counts(gen_counts, dominance='classical'):
    phen_counts = Counter()
    for g,c in gen_counts.items():
        phen = phenotype_dihybrid(g, dominance=dominance)
        phen_counts[phen] += c
    return phen_counts

def exact_cross_counts(parent1_genotype, parent2_genotype):
    gam1 = [''.join(g) for g in gametes_from_parent(parent1_genotype)]
    gam2 = [''.join(g) for g in gametes_from_parent(parent2_genotype)]
    offspring = []
    for a in gam1:
        for b in gam2:
            loci_pairs = []
            for x,y in zip(a,b):
                loci_pairs.append(''.join(sorted([x,y], key=lambda z: z.islower())))
            offspring.append(''.join(loci_pairs))
    counts = Counter(offspring)
    total = sum(counts.values())
    ratios = {k: v/total for k,v in counts.items()}
    return counts, ratios


In [ ]:
#@title Run monohybrid simulation { display-mode: "form" }
parent1 = "Aa" #@param {type:"string"}
parent2 = "Aa" #@param {type:"string"}
n_offspring = 5000 #@param {type:"integer"}

counts = simulate_monohybrid(parent1, parent2, n_offspring=n_offspring)
print("Counts:", counts)
print("\nPunnett square:")
display(punnett_square(parent1, parent2))


In [ ]:
#@title Monohybrid chi-square test { display-mode: "form" }
# Uses the 'counts' variable from previous cell
observed_counts = [counts.get('AA',0), counts.get('Aa',0), counts.get('aa',0)]
total = sum(observed_counts)
expected_ratio = [1,2,1]
expected_counts = [r/sum(expected_ratio)*total for r in expected_ratio]

chisq, dof = chi_square(observed_counts, expected_counts)
print("Observed (AA, Aa, aa):", observed_counts)
print("Expected (AA, Aa, aa):", [round(x,1) for x in expected_counts])
print("Chi-square:", round(chisq,3), " df:", dof)


In [ ]:
#@title Run dihybrid simulation { display-mode: "form" }
parent1_d = "AaBb" #@param {type:"string"}
parent2_d = "AaBb" #@param {type:"string"}
n_offspring_d = 20000 #@param {type:"integer"}

d_counts = simulate_dihybrid(parent1_d, parent2_d, n_offspring=n_offspring_d)
print("Top genotype counts (sample):", d_counts.most_common(8))
print("\nPunnett square (4x4):")
display(punnett_square_dihybrid(parent1_d, parent2_d))


In [ ]:
#@title Dihybrid phenotype chi-square test { display-mode: "form" }
dominance_mode = "classical" #@param ["classical", "incomplete"]
phen_counts = phenotype_counts_from_genotype_counts(d_counts, dominance=dominance_mode)

total = sum(phen_counts.values())
expected_ratios = {
    'Dominant_Dominant': 9,
    'Dominant_Recessive': 3,
    'Recessive_Dominant': 3,
    'Recessive_Recessive': 1
}
obs = [
    phen_counts.get('Dominant_Dominant', 0),
    phen_counts.get('Dominant_Recessive', 0),
    phen_counts.get('Recessive_Dominant', 0),
    phen_counts.get('Recessive_Recessive', 0)
]
expected = [expected_ratios[k]/16 * total for k in expected_ratios]
chisq, dof = chi_square(obs, expected)

print("Phenotype counts:", phen_counts)
print("\nObserved:", obs)
print("Expected:", [round(x,1) for x in expected])
print("Chi-square:", round(chisq,3), " df:", dof)


In [ ]:
#@title Exact cross counts (deterministic) { display-mode: "form" }
parent1_exact = "AaBb" #@param {type:"string"}
parent2_exact = "AaBb" #@param {type:"string"}

exact_counts, exact_ratios = exact_cross_counts(parent1_exact, parent2_exact)
print("Exact counts (16 outcomes):")
for k,v in sorted(exact_counts.items()):
    print(k, v)
print("\nExact ratios:")
for k,v in sorted(exact_ratios.items()):
    print(k, '{:.3f}'.format(v))



## Exercises

1. Use the form controls above to run different parent combinations and sample sizes.
2. Compare `simulate_dihybrid` sampling results with `exact_cross_counts` ratios by multiplying ratios by the sample size.
3. Try incomplete dominance by setting dominance mode to 'incomplete' and designing expected ratios for exercises.
4. Create a classroom dataset and run the chi-square tests here.
